<a href="https://colab.research.google.com/github/pinkadace-code/llm-context-generator/blob/main/llm_context_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mysql-connector-python google-generativeai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 44.8 MB/s eta 0:00:00


In [2]:
import mysql.connector

# Savienojuma parametri
conn = mysql.connector.connect(
    host="87.110.123.151",
    port=3306,
    user="fita",
    password="2026-04-28"
)

cursor = conn.cursor()

# Pārbaudām savienojumu
cursor.execute("SHOW DATABASES;")
databases = cursor.fetchall()

print("✅ Savienojums veiksmīgs!")
print("\nAtrastās datubāzes:")
for db in databases:
    print(f"  📁 {db[0]}")

✅ Savienojums veiksmīgs!

Atrastās datubāzes:
  📁 direct_payments
  📁 information_schema
  📁 performance_schema


In [3]:
# Iegūstam visu struktūru no direct_payments datubāzes
cursor.execute("USE direct_payments;")

# Iegūstam visas tabulas
cursor.execute("SHOW TABLES;")
tables = [row[0] for row in cursor.fetchall()]

print(f"📊 Atrastas {len(tables)} tabulas:\n")

# Katrai tabulai iegūstam kolonnu info
db_structure = {}

for table in tables:
    cursor.execute(f"DESCRIBE `{table}`;")
    columns = cursor.fetchall()

    db_structure[table] = []
    print(f"📋 Tabula: {table}")
    print(f"{'Kolonna':<30} {'Tips':<20} {'Nullable':<10} {'Atslēga':<10}")
    print("-" * 70)

    for col in columns:
        name, dtype, nullable, key, default, extra = col
        db_structure[table].append({
            "nosaukums": name,
            "tips": dtype,
            "nullable": nullable,
            "atslega": key,
            "noklusejums": default,
            "extra": extra
        })
        print(f"{name:<30} {dtype:<20} {nullable:<10} {key:<10}")

    print()

print("✅ Struktūra iegūta veiksmīgi!")

📊 Atrastas 3 tabulas:

📋 Tabula: mandates
Kolonna                        Tips                 Nullable   Atslēga   
----------------------------------------------------------------------
id                             varchar(50)          YES                  
created_at                     datetime             YES                  
scheme                         varchar(50)          YES                  
organisation_id                varchar(50)          YES                  

📋 Tabula: organisations
Kolonna                        Tips                 Nullable   Atslēga   
----------------------------------------------------------------------
id                             varchar(50)          YES                  
created_at                     datetime             YES                  
parent_vertical                varchar(50)          YES                  

📋 Tabula: payments
Kolonna                        Tips                 Nullable   Atslēga   
-------------------------------

In [4]:
# Izveidojam teksta kontekstu ko nosūtīsim Gemini
context = """
Tu strādā ar MySQL datubāzi 'direct_payments'. Tajā ir 3 tabulas:

1. TABULA: organisations
   - id (varchar) — organizācijas unikālais identifikators
   - created_at (datetime) — kad organizācija pievienojās
   - parent_vertical (varchar) — organizācijas nozare/kategorija

2. TABULA: mandates
   - id (varchar) — pilnvaras unikālais identifikators
   - created_at (datetime) — kad pilnvara izveidota
   - scheme (varchar) — maksājumu shēma (piemēram, BACS, SEPA)
   - organisation_id (varchar) — saite uz organisations.id

3. TABULA: payments
   - id (varchar) — maksājuma unikālais identifikators
   - amount (double) — maksājuma summa
   - currency (varchar) — valūta (piemēram, GBP, EUR)
   - created_at (datetime) — kad maksājums izveidots
   - source (varchar) — maksājuma avots
   - charge_date (datetime) — kad maksājums tika iekasēts
   - mandate_id (varchar) — saite uz mandates.id

SAISTĪBAS:
- organisations.id = mandates.organisation_id
- mandates.id = payments.mandate_id

Lai iegūtu organizācijas maksājumus, jāsavieno visas 3 tabulas.
"""

print(context)
print("✅ Konteksts izveidots!")


Tu strādā ar MySQL datubāzi 'direct_payments'. Tajā ir 3 tabulas:

1. TABULA: organisations
   - id (varchar) — organizācijas unikālais identifikators
   - created_at (datetime) — kad organizācija pievienojās
   - parent_vertical (varchar) — organizācijas nozare/kategorija

2. TABULA: mandates
   - id (varchar) — pilnvaras unikālais identifikators
   - created_at (datetime) — kad pilnvara izveidota
   - scheme (varchar) — maksājumu shēma (piemēram, BACS, SEPA)
   - organisation_id (varchar) — saite uz organisations.id

3. TABULA: payments
   - id (varchar) — maksājuma unikālais identifikators
   - amount (double) — maksājuma summa
   - currency (varchar) — valūta (piemēram, GBP, EUR)
   - created_at (datetime) — kad maksājums izveidots
   - source (varchar) — maksājuma avots
   - charge_date (datetime) — kad maksājums tika iekasēts
   - mandate_id (varchar) — saite uz mandates.id

SAISTĪBAS:
- organisations.id = mandates.organisation_id
- mandates.id = payments.mandate_id

Lai iegūtu 

In [6]:
!pip install -q google-genai

In [12]:
# Apskatām kādi modeļi ir pieejami
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [15]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.1 MB/s eta 0:00:00


In [16]:
from groq import Groq
from google.colab import userdata

# Pieslēdzamies Groq
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Funkcija SQL ģenerēšanai
def generate_sql(jautajums):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": f"""Tu esi SQL eksperts. Šeit ir datubāzes struktūra:

{context}

Atgriezī TIKAI SQL kodu bez jebkādiem paskaidrojumiem vai markdown formatējuma."""
            },
            {
                "role": "user",
                "content": jautajums
            }
        ]
    )
    return response.choices[0].message.content.strip()

# Testējam
jautajums = "Cik maksājumu ir katrā nozarē (parent_vertical)?"
sql = generate_sql(jautajums)

print(f"❓ Jautājums: {jautajums}")
print(f"\n📝 Ģenerētais SQL:\n{sql}")

❓ Jautājums: Cik maksājumu ir katrā nozarē (parent_vertical)?

📝 Ģenerētais SQL:
SELECT o.parent_vertical, COUNT(p.id) AS skaitlis 
FROM organisations o 
JOIN mandates m ON o.id = m.organisation_id 
JOIN payments p ON m.id = p.mandate_id 
GROUP BY o.parent_vertical


In [17]:
import pandas as pd

def execute_sql(sql):
    cursor.execute(sql)
    results = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(results, columns=columns)
    return df

# Izpildām ģenerēto SQL
df = execute_sql(sql)

print("📊 Rezultāti:")
print(df.to_string(index=False))

📊 Rezultāti:
                         parent_vertical  skaitlis
                                property      2545
tradesmen_and_non_professionals_services      8080
                          sports_fitness     12170
         digital_services_media_telecoms      3995
     professional_and_financial_services      5126
                     societies_and_clubs       228
                              healthcare      1317


In [18]:
def describe_data(jautajums, sql, df):
    dati = df.to_string(index=False)

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Tu esi datu analītiķis. Apraksti datus latviešu valodā — skaidri, kodolīgi un ar konkrētiem skaitļiem."
            },
            {
                "role": "user",
                "content": f"""
Jautājums: {jautajums}

SQL vaicājums: {sql}

Dati:
{dati}

Apraksti ko šie dati rāda, kādas ir galvenās atziņas.
"""
            }
        ]
    )
    return response.choices[0].message.content.strip()

# Testējam
apraksts = describe_data(jautajums, sql, df)
print("🤖 AI apraksts:")
print(apraksts)

🤖 AI apraksts:
Šie dati rāda maksājumu skaits katrā nozarē, ko nosauc par "parent_vertical". Kopumā tie sniedz informāciju par to, cik dažādās nozarēs tiek veikti maksājumi.

Galvenās atziņas šo datu analīzē ir:

1. **Visvairāk maksājumu ir sports un fiziskās sagatavotības nozarē**: 12 170 maksājumi, kas rāda, ka šī nozare ir visaktīvākā maksājumu ziņā.
2. **Amatnieku un neprofesionālo pakalpojumu sniegšanas nozare ir otra aktivākā**: 8 080 maksājumi, kas norāda uz to, ka šajā nozarē tiek veikti daudzi maksājumi.
3. **Profesionālo un finanšu pakalpojumu nozare turpmāk seko ar 5 126 maksājumiem**: Šī nozare ir svarīga saistībā ar finanšu darījumiem un profesionālo pakalpojumu sniegšanu.
4. **Citskaitļa nozares, kā digitālie pakalpojumi, medicīna un sabiedrības un klubu nozare**, arī rāda nozīmīgu maksājumu skaitu, taču tas ir mazāks salīdzinot ar visaktīvākajām nozarēm.

Šie dati varētu palīdzēt izprast maksājumu modeļus dažādās nozarēs un norādīt uz iespējamām tendencēm un attīstības p